# OOP e Reti Neurali con PyTorch

In questa lezione vedremo come i principi della **programmazione orientata agli oggetti** (OOP) siano il fondamento su cui PyTorch costruisce le reti neurali.

**Indice:**
1. Richiamo OOP — classi e oggetti
2. Ereditarietà — il meccanismo chiave
3. `torch.nn.Module` — la classe base di ogni rete
4. Costruire la prima rete neurale
5. Anatomia di un modello: parametri e stato
6. Funzioni di attivazione
7. Composizione di moduli — `nn.Sequential`
8. Loss function e Optimizer (anch'essi oggetti)
9. Il training loop completo
10. Salvare e caricare un modello

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)  # riproducibilità
print(f"PyTorch {torch.__version__}")

---
## 1. Richiamo OOP — classi e oggetti

Una **classe** è un modello (template). Un **oggetto** è un'istanza concreta di quella classe.

Ogni oggetto ha:
- **Attributi**: dati che descrivono lo stato dell'oggetto
- **Metodi**: funzioni che operano su quegli attributi

In [ ]:
class Neurone:
    """Un singolo neurone con pesi e bias."""

    def __init__(self, n_input):
        # Attributi: stato interno dell'oggetto
        self.pesi = torch.randn(n_input)
        self.bias = torch.zeros(1)

    def forward(self, x):
        """Calcola l'output: combinazione lineare + attivazione."""
        z = torch.dot(self.pesi, x) + self.bias
        return torch.sigmoid(z)  # attivazione sigmoide

    def __repr__(self):
        return f"Neurone(n_input={len(self.pesi)})"


# Creare due istanze distinte — ognuna ha i propri pesi
n1 = Neurone(3)
n2 = Neurone(3)

x = torch.tensor([0.5, -1.0, 2.0])
print(n1)                          # usa __repr__
print("Output n1:", n1.forward(x))
print("Output n2:", n2.forward(x)) # diverso! pesi diversi

---
## 2. Ereditarietà — il meccanismo chiave

Con l'**ereditarietà** una classe figlia acquisisce attributi e metodi della classe padre, potendoli estendere o ridefinire.

```
nn.Module          ← classe padre (PyTorch)
    └── MiaRete    ← classe figlia (la nostra rete)
```

Ereditare da `nn.Module` non è una formalità: **attiva l'intera infrastruttura PyTorch**.

In [ ]:
# Esempio didattico: ereditarietà semplice
class Animale:
    def __init__(self, nome):
        self.nome = nome

    def parla(self):
        return "..."

class Cane(Animale):            # Cane EREDITA da Animale
    def parla(self):            # override del metodo
        return "Bau!"

class Gatto(Animale):
    def parla(self):
        return "Miao!"

animali = [Cane("Rex"), Gatto("Felix")]
for a in animali:
    print(f"{a.nome} dice: {a.parla()}")  # polimorfismo

> 💡 **Polimorfismo**: lo stesso metodo `.parla()` si comporta diversamente a seconda del tipo di oggetto. In PyTorch, lo stesso meccanismo vale per `.forward()` — ogni rete definisce la propria logica di inferenza.

---
## 3. `torch.nn.Module` — la classe base di ogni rete

In PyTorch, **ogni rete neurale è una sottoclasse di `nn.Module`**.

Regole da rispettare:
1. Definire `__init__` e chiamare `super().__init__()` come **prima cosa**
2. Definire `forward(self, x)` con la logica di inferenza
3. Dichiarare i layer in `__init__` (così PyTorch li registra automaticamente)

In [ ]:
class ReteSemplice(nn.Module):

    def __init__(self, n_input, n_hidden, n_output):
        super().__init__()              # ← OBBLIGATORIO: inizializza nn.Module

        # Attributi: i layer sono oggetti nn.Linear registrati automaticamente
        self.layer1 = nn.Linear(n_input, n_hidden)
        self.layer2 = nn.Linear(n_hidden, n_output)

    def forward(self, x):              # ← definisce il calcolo
        x = torch.relu(self.layer1(x))
        x = self.layer2(x)
        return x


# Istanziare il modello
modello = ReteSemplice(n_input=4, n_hidden=8, n_output=2)
print(modello)  # PyTorch stampa l'architettura automaticamente

In [ ]:
# Invocare il modello — si usa come una funzione, NON .forward() direttamente
# (PyTorch esegue hooks importanti in __call__, che chiama forward internamente)
x = torch.randn(1, 4)  # 1 campione, 4 feature
output = modello(x)
print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print("Output:      ", output)

> Usa sempre `modello(x)` e **mai** `modello.forward(x)` direttamente.

---
## 4. Costruire la prima rete neurale

Costruiamo un classificatore binario per il problema XOR — non linearmente separabile, quindi richiede almeno un layer nascosto.

In [ ]:
# Dataset XOR
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])  # output XOR

print("Input X:\n", X)
print("Target y:\n", y)

In [ ]:
class XORNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)   # 2 input → 4 neuroni nascosti
        self.fc2 = nn.Linear(4, 1)   # 4 nascosti → 1 output

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # attivazione ReLU nel layer nascosto
        x = torch.sigmoid(self.fc2(x))  # sigmoide: output in [0, 1]
        return x


net = XORNet()
print(net)

# Prova forward pass prima del training
with torch.no_grad():
    pred = net(X)
print("\nPredizioni iniziali (random):\n", pred)

---
## 5. Anatomia di un modello: parametri e stato

In [ ]:
# .parameters() — tutti i tensori apprendibili (pesi e bias)
print("=== Parametri del modello ===")
for nome, param in net.named_parameters():
    print(f"  {nome:15s} shape={str(param.shape):20s} requires_grad={param.requires_grad}")

# Contare i parametri totali
totale = sum(p.numel() for p in net.parameters())
print(f"\nParametri totali: {totale}")

In [ ]:
# .state_dict() — dizionario con tutti i pesi (usato per salvare/caricare)
print("=== State dict ===")
for chiave, valore in net.state_dict().items():
    print(f"  {chiave:15s}: {valore}")

---
## 6. Funzioni di attivazione

Le funzioni di attivazione introducono **non linearità** — senza di esse, una rete con N layer sarebbe equivalente a un singolo layer lineare.

In [ ]:
x = torch.linspace(-4, 4, 100)

attivazioni = {
    "ReLU":    torch.relu(x),
    "Sigmoid": torch.sigmoid(x),
    "Tanh":    torch.tanh(x),
    "GELU":    nn.functional.gelu(x),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (nome, y) in zip(axes, attivazioni.items()):
    ax.plot(x.numpy(), y.detach().numpy(), lw=2, color="steelblue")
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.axvline(0, color="gray", lw=0.8, ls="--")
    ax.set_title(nome, fontsize=13)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Quando usarle:")
print("  ReLU    → layer nascosti (default, veloce)")
print("  Sigmoid → output binario (classificazione)")
print("  Tanh    → output in [-1, 1], RNN")
print("  GELU    → Transformer, modelli moderni")

---
## 7. Composizione di moduli — `nn.Sequential`

Un modello può **contenere altri moduli** come attributi.
`nn.Sequential` è il modo più compatto per reti feed-forward lineari.

In [ ]:
# Modo 1: nn.Sequential (compatto, per architetture lineari)
modello_seq = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)
print("Sequential:\n", modello_seq)

In [ ]:
# Modo 2: classe personalizzata (flessibile, per architetture complesse)
# Utile quando servono skip connections, branch multipli, ecc.
class ReteProfonda(nn.Module):

    def __init__(self, n_input, layers, n_output):
        """Costruisce dinamicamente un MLP con n layers nascosti."""
        super().__init__()

        # nn.ModuleList: lista di moduli registrata correttamente
        dims = [n_input] + layers + [n_output]
        self.layers = nn.ModuleList([
            nn.Linear(dims[i], dims[i+1])
            for i in range(len(dims)-1)
        ])

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = torch.relu(layer(x))
        return self.layers[-1](x)  # ultimo layer senza attivazione


rete = ReteProfonda(n_input=4, layers=[16, 8, 4], n_output=2)
print(rete)
print(f"\nParametri totali: {sum(p.numel() for p in rete.parameters())}")

---
## 8. Loss function e Optimizer — anch'essi oggetti

In [ ]:
# Loss functions comuni
loss_bce  = nn.BCELoss()           # Binary Cross Entropy (classificazione binaria)
loss_ce   = nn.CrossEntropyLoss()  # Cross Entropy (classificazione multiclasse)
loss_mse  = nn.MSELoss()           # Mean Squared Error (regressione)
loss_mae  = nn.L1Loss()            # Mean Absolute Error

# Esempio: calcolare la loss
y_pred = torch.tensor([0.9, 0.1, 0.8, 0.3])  # probabilità predette
y_true = torch.tensor([1.0, 0.0, 1.0, 0.0])  # etichette reali
print("BCE Loss:", loss_bce(y_pred, y_true).item())
print("MSE Loss:", loss_mse(y_pred, y_true).item())

In [ ]:
# Optimizer — aggiorna i pesi in base ai gradienti
net = XORNet()

# L'optimizer riceve i parametri del modello come oggetto
sgd  = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)
adam = optim.Adam(net.parameters(), lr=0.001)

print("SGD: ", sgd)
print("Adam:", adam)

---
## 9. Il training loop completo

Tutto si unisce qui. Il loop di addestramento ha sempre la stessa struttura:

```
per ogni epoca:
  1. Forward pass  → calcola le predizioni
  2. Calcola loss  → misura l'errore
  3. Zero grad     → azzera i gradienti accumulati
  4. Backward      → calcola i gradienti (backprop)
  5. Step          → aggiorna i pesi
```

In [ ]:
# Setup
net       = XORNet()
optimizer = optim.Adam(net.parameters(), lr=0.05)
criterion = nn.BCELoss()

losses = []
N_EPOCHE = 500

# Training loop
for epoca in range(N_EPOCHE):

    # 1. Forward pass
    y_pred = net(X)

    # 2. Calcola loss
    loss = criterion(y_pred, y)

    # 3. Azzera i gradienti (IMPORTANTE: si accumulano altrimenti)
    optimizer.zero_grad()

    # 4. Backpropagation
    loss.backward()

    # 5. Aggiorna i pesi
    optimizer.step()

    losses.append(loss.item())

    if (epoca + 1) % 100 == 0:
        print(f"Epoca {epoca+1:4d} | Loss: {loss.item():.4f}")

print("\nAddestramento completato!")

In [ ]:
# Curva di loss
plt.figure(figsize=(8, 3))
plt.plot(losses, color="steelblue", lw=2)
plt.xlabel("Epoca")
plt.ylabel("Loss (BCE)")
plt.title("Curva di addestramento — XOR")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Valutazione finale
net.eval()  # modalità inferenza (disabilita dropout, batchnorm, ecc.)
with torch.no_grad():
    predizioni = net(X)
    classi     = (predizioni > 0.5).float()

print("Input   | Target | Predizione | Classe")
print("-" * 45)
for xi, yi, pi, ci in zip(X, y, predizioni, classi):
    print(f"{xi.tolist()}  |   {int(yi.item())}   |   {pi.item():.3f}    |   {int(ci.item())}")

accuracy = (classi == y).float().mean()
print(f"\nAccuracy: {accuracy.item()*100:.1f}%")

---
## 10. Salvare e caricare un modello

In [ ]:
# Salvare solo i pesi (raccomandato)
torch.save(net.state_dict(), "xor_model.pth")
print("Modello salvato in xor_model.pth")

# Caricare i pesi in una nuova istanza
net_caricato = XORNet()                              # stessa architettura
net_caricato.load_state_dict(torch.load("xor_model.pth", weights_only=True))
net_caricato.eval()

print("Modello caricato con successo!")

# Verifica: predizioni identiche
with torch.no_grad():
    p_nuovo = net_caricato(X)
    print("\nPredizioni (modello ricaricato):")
    print(p_nuovo)

---
## Riepilogo — OOP e Reti Neurali

| Concetto OOP | Equivalente in PyTorch |
|:---|:---|
| **Classe** | Definizione dell'architettura (`class MiaRete(nn.Module)`) |
| **Oggetto** | Istanza della rete (`net = MiaRete()`) |
| **Attributi** | Layer, pesi, bias (`self.fc1 = nn.Linear(...)`) |
| **Metodo** | `forward(x)` — logica di inferenza |
| **Ereditarietà** | `nn.Module` fornisce parametri, gradienti, salvataggio |
| **Polimorfismo** | Ogni rete ha il suo `forward`, chiamato da `modello(x)` |
| **Composizione** | Una rete contiene altri moduli (`nn.ModuleList`, `nn.Sequential`) |

---

### Il training loop in 5 righe
```python
y_pred = net(X)           # 1. forward
loss   = criterion(y_pred, y)  # 2. loss
optimizer.zero_grad()     # 3. azzera gradienti
loss.backward()           # 4. backprop
optimizer.step()          # 5. aggiorna pesi
```

---

### Prossimi passi
- Dati reali con `Dataset` e `DataLoader`
- Reti convoluzionali (`nn.Conv2d`) per immagini
- Tecniche di regolarizzazione: Dropout, BatchNorm
- Transfer learning e modelli pre-addestrati
